# Validation of ERA5-Land in the Magdalena River Basin (daily): standard metrics

Classical errors (MAE, MSE, RMSE, correlation, mutual information, cross entropy) between ERA5-Land, the IDW-interpolated rain gauges and the local stations. The calculations live in `snlda.metrics`; this notebook only loads the data, calls them and saves the results.

## 0. Setup

In [ ]:
from pathlib import Path

DATA = Path("Data")
RESULTS = Path("Results")

STATIONS_CSV = DATA / "Local CSV" / "Precipitation 1980-2020 data filled 1.csv"
COORDS_CSV = DATA / "Local CSV" / "Precipitation 1980-2020 coord filled.csv"
IDW_NC = DATA / "Local IDW" / "IDW_daily_010.nc"
ERA_NC = DATA / "ERALand NC" / "precipitation_eraland_d.nc"
BASIN_NC = DATA / "Magdalena" / "GIS" / "Basin" / "Basin_ext_010.nc"

import pandas as pd

from snlda import grid, io, metrics

## 1. Load data

In [ ]:
stations, coords = io.read_station_data(STATIONS_CSV, COORDS_CSV)
idw = io.read_netcdf_grid(IDW_NC)      # local gauges interpolated with IDW, 0.1 deg
era = io.read_netcdf_grid(ERA_NC)      # ERA5-Land, 0.1 deg
basin = io.read_basin_mask(BASIN_NC)

datasets = {"IDW 010": idw, "ERA5": era}

## 2. Error to local stations

In [ ]:
out_dir = RESULTS / "Error to local stations"
out_dir.mkdir(parents=True, exist_ok=True)

station_errors = {}
for name, ds in datasets.items():
    errors = metrics.station_metrics(stations, coords, ds.data, ds.lat, ds.lon)
    errors.to_csv(out_dir / f"{name}_daily.csv")
    station_errors[name] = errors

station_errors["ERA5"].head()

## 3. Error between ERA5-Land and the IDW interpolation (per pixel)

In [ ]:
# Put both datasets on the basin grid (cells missing from a dataset become NaN)
idw_basin = grid.select_on_grid(idw.data, idw.lat, idw.lon, basin.lat, basin.lon)
era_basin = grid.select_on_grid(era.data, era.lat, era.lon, basin.lat, basin.lon)

pixel_errors = metrics.pixel_metrics(idw_basin, era_basin, basin.mask)

In [ ]:
out_dir = Path("Error to interpolations")
out_dir.mkdir(parents=True, exist_ok=True)

for metric, array in pixel_errors.items():
    io.write_netcdf_grid(
        out_dir / f"{metric}_daily_010.nc",
        {metric: array},
        lat=basin.lat,
        lon=basin.lon,
        title=metric,
    )